# 02/ Test DESC SN Ia metric SNNSNMMetrics : Get Number of SNeIa

- **try with `sne_nside = 16, 8` and without and with dust**

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-05
- last update : 2026-08-09
- copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/science/Number_SNeIa_metric.ipynb

**What this Metrics `SNNSNMetric`) does**

`SNNSNMetric` (contributed by the DESC/Philippe Gris group) is a *cadence metric*: instead of just
counting visits, it asks a survey-strategy question — "given the actual sequence of LSST visits
(nights, filters, depths) at a given point on the sky, how well could we discover and characterize
Type Ia supernovae there?"

For each sky pixel it internally does the following, using ONLY the simulated visits that fall in that pixel:
1. Generate a grid of fake SNe Ia at different redshifts and explosion (peak) dates.
2. For each fake SN, check which of the real LSST visits would have observed it before/after peak
   brightness, and estimate the photometric uncertainty of each point (using a gamma/noise model).
3. Fit a simplified SALT2-like light curve and evaluate the color-uncertainty (sigma_color).
4. Find `zlim`: the highest redshift at which enough SNe pass quality cuts (sigma_color threshold,
   minimum epochs before/after peak) to be usable for cosmology.
5. Combine `zlim` with the SN Ia volumetric rate to get `nSN`: the expected number of well-measured
   SNe Ia out to that redshift, for that field/pixel and survey duration.


**What this notebook does**

Where `01_testSNIa.ipynb` dissects the metric on a single pixel, this notebook focuses on the final
science output: the **number of SNe Ia** (`nSN`) the survey is expected to yield.

It runs `SNNSNMetric` with default parameters at two different HEALPix resolutions (`nside=16` then
`nside=8`) to check how sensitive the summary numbers are to pixel size, and finally cross-checks the
metric's built-in rate against `SnRate`, the standalone DESC SN Ia volumetric rate model used inside
the metric to convert `zlim` (redshift completeness limit) into `nSN` (expected SN count).

Recap of the two key outputs produced per HEALPix pixel by `SNNSNMetric`:
- `zlim`: highest redshift at which enough simulated SNe Ia pass the light-curve quality cuts
  (minimum epochs before/after peak, color-uncertainty threshold) to be usable for cosmology.
- `n_sn`: number of SNe Ia expected out to `zlim`, obtained by integrating the SN Ia volumetric rate
  (`SnRate`, explored at the end of this notebook) over the pixel's solid angle and the survey duration.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
import rubin_sim.maf as maf
# import rubin_sim.utils as rsUtils

# get_baseline() moved between packages across rubin_sim versions; try both import paths.
try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

import healpy as hp

## 1. Configuration

In [ ]:
# RUBIN_SIM_DATA_DIR points to the local cache of rubin_sim/rubin_scheduler auxiliary data
# (opsim databases, dust maps, SN gamma/noise files, throughputs, ...).
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
# Baseline Survey
# Reference LSST wide-fast-deep (WFD) cadence simulation used for this notebook's metric runs.
baseline_file = get_baseline()
runname = baseline_file.split("/")[-1].replace(".db", "")
print(runname)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    # Scratch directory for MAF's results database and any output products from this notebook.
    data_dir_itself = tempfile.TemporaryDirectory(prefix="02_maf_Number_SNIa_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

In [ ]:
# Set up MAF output
# ResultsDb is MAF's bookkeeping database that tracks which metric bundles were run/plotted here.
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

## 2. SNNSNMMetrics NSIDE = 16 - No Dust

### 2.1 Define the slicer, metrics and bundle for NSIDE = 16 - No dust

In [ ]:
bundle_list = []

# HEALPix resolution: nside=16 -> 12*16^2 = 3072 pixels over the sky.
sne_nside = 16
sqlconstraint = ""  # no SQL filtering on visits: use the full baseline cadence as-is
# Summary metrics collapse the per-pixel skymap into single sky-averaged numbers.
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]
slicer = maf.slicers.HealpixSlicer(nside=sne_nside, use_cache=False)
# SNNSNMetric with default light-curve/redshift parameters, dust extinction disabled here.
metric = maf.metrics.SNNSNMetric(verbose=False, add_dust=False)  # zlim_coeff=0.98)
bundle = maf.metric_bundle.MetricBundle(
    metric, slicer, sqlconstraint, summary_metrics=sn_summary, run_name=runname
)

### 2.2 Create group bundle and run it

In [ ]:
# Attach the bundle to the baseline opsim database so run_all() knows which visits to use.
bdict = {"sne": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

In [ ]:
# Run SNNSNMetric on every HEALPix pixel (nside=16), then plot the resulting sky maps (n_sn, zlim).
group.run_all()

### 2.4 Plot Metrics results

In [ ]:
group.plot_all(closefigs=False)

### 2.5 Sky-averaged summary values for SSIDE = 16 and without dust extinction

In [ ]:
# Sky-averaged summary values (median/sum/mean, per sn_summary above) for the two reduced
# quantities: expected SN Ia count (n_sn) and redshift completeness limit (zlim).
bdict["SNNSNMetric_reducen_sn"].summary_values, bdict["SNNSNMetric_reducezlim"].summary_values

In [ ]:
print("Sky-averaged summary values at nside = 16 (No dust) : N_SN")
display(pd.Series(bdict["SNNSNMetric_reducen_sn"].summary_values))

In [ ]:
print("Sky-averaged summary values at nside = 16 (No dust) : N_ZLIM")
display(pd.Series(bdict["SNNSNMetric_reducezlim"].summary_values))

In [ ]:
for k, val in bdict.items():
    display(pd.Series(val.summary_values, name=k))

## 3. SNNSNMMetrics NSIDE = 8 - No Dust

### 3.1 Define the metrics and bundle for NSIDE = 8 and NO add_dust = False

In [ ]:
bundle_list = []

# Repeat the same setup at coarser resolution (nside=8 -> 768 pixels) to see how much the
# sky-averaged n_sn/zlim summary values depend on the chosen HEALPix pixel size.
sne_nside = 8
sql = ""
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]
slicer = maf.slicers.HealpixSlicer(nside=sne_nside, use_cache=False)
metric = maf.metrics.SNNSNMetric(verbose=False, add_dust=False)  # zlim_coeff=0.98)
bundle = maf.metric_bundle.MetricBundle(metric, slicer, sql, summary_metrics=sn_summary, run_name=runname)

### 3.2 Create group bundle and run it

In [ ]:
bdict = {"sne": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)
group.run_all()

In [ ]:
group.plot_all(closefigs=False)

### 3.3 Sky-averaged summary values with NO dust extinction

In [ ]:
# Sky-averaged summary values at nside=8, to compare with the nside=16 run above.
bdict["SNNSNMetric_reducen_sn"].summary_values, bdict["SNNSNMetric_reducezlim"].summary_values

In [ ]:
print("Sky-averaged summary values at nside = 8 (No dust) : N_SN")
display(pd.Series(bdict["SNNSNMetric_reducen_sn"].summary_values))

In [ ]:
print("Sky-averaged summary values at nside = 8 (No dust) : N_ZLIM")
display(pd.Series(bdict["SNNSNMetric_reducezlim"].summary_values))

In [ ]:
for k, val in bdict.items():
    display(pd.Series(val.summary_values, name=k))

## 4. SNNSNMMetrics NSIDE = 8 - with Dust Extintion

### 4.1 Define the metrics and bundle for NSIDE = 8 and add_dust = True

In [ ]:
bundle_list = []

# Same nside=8 setup, but with add_dust left at its default (True) instead of explicitly False,
# so Milky Way extinction IS applied this time -- compare the resulting n_sn/zlim to the run above.
# Note: 'runname=runname' below is a typo for 'run_name=runname' (kept as-is from the original
# notebook); MetricBundle will simply ignore the unrecognized keyword rather than raise an error.
sne_nside = 8
sql = ""
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]
slicer = maf.slicers.HealpixSlicer(nside=sne_nside, use_cache=False)
metric = maf.metrics.SNNSNMetric(verbose=False)  # zlim_coeff=0.98)
bundle = maf.metric_bundle.MetricBundle(metric, slicer, sql, summary_metrics=sn_summary, run_name=runname)

bdict = {"sne": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)
group.run_all()
group.plot_all(closefigs=False)

### 4.2 Create group bundle and run it

In [ ]:
bdict = {"sne": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)
group.run_all()

### 4.4 Plot Metrics figures

In [ ]:
group.plot_all(closefigs=False)

### 4.5  Sky-averaged summary values with dust extinction enabled 

In [ ]:
bdict = group.bundle_dict.copy()

In [ ]:
# Sky-averaged summary values with dust extinction enabled -- typically lower n_sn/zlim than
# the add_dust=False runs above, since dust-reddened lines of sight lose some SNe to the quality cuts.
bdict["SNNSNMetric_reducen_sn"].summary_values, bdict["SNNSNMetric_reducezlim"].summary_values

In [ ]:
print("Sky-averaged summary values at nside = 8 (with dust) : N_SN")
display(pd.Series(bdict["SNNSNMetric_reducen_sn"].summary_values))

In [ ]:
print("Sky-averaged summary values at nside = 8 (with dust) : N_ZLIM")
display(pd.Series(bdict["SNNSNMetric_reducezlim"].summary_values))

In [ ]:
for k, val in bdict.items():
    display(pd.Series(val.summary_values, name=k))

## 5. Supernovae rate

`SnRate` is the standalone SN Ia volumetric rate model that `SNNSNMetric` uses internally to convert
a redshift completeness limit (`zlim`) into an expected SN count (`nSN`). Calling it directly here
(outside of MAF) lets us sanity-check the rate model and see how it depends on the assumed survey
area (`survey_area`, in square degrees).

In [ ]:
from rubin_sim.maf.utils.sn_n_sn_utils import SnRate

In [ ]:
# Instantiate the SN Ia rate calculator with its default cosmology/rate-model parameters.
ack = SnRate()

In [ ]:
# Call it with default survey area to get the expected SN Ia counts (e.g. vs. redshift bin).
ack()

In [ ]:
# Same rate calculation but scaled to a smaller survey area (9.6/2.0 sq deg), e.g. roughly the
# area of a single Deep Drilling Field pointing, to see how nSN scales with the area covered.
ack(survey_area=9.6 / 2.0)